<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L32_Custom_Architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L32: Custom Architectures - Attention and Transformer Variants

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 32 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Implement a minimal custom attention layer and use it in a small model
- Understand design choices: heads, dimensions, normalization
- Experiment with simple architectural variants

---

## Concept: Custom Architectures

Custom transformers let you change: attention (e.g. linear attention, sparse patterns), normalization (Pre-LN vs Post-LN), activation, or add new components. We build a small custom multi-head self-attention block and a 1-layer "transformer" block.

---

In [ ]:
import torch
import torch.nn as nn
import math
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Custom Multi-Head Self-Attention

---

In [ ]:
class CustomMultiHeadAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)

attn = CustomMultiHeadAttention(256, 4)
x = torch.randn(2, 10, 256)
y = attn(x)
print("Custom attention output shape:", y.shape)

## Part 2: Simple Transformer Block (Attention + FFN)

---

In [ ]:
class CustomTransformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, d_ff=1024, dropout=0.1):
        super().__init__()
        self.attn = CustomMultiHeadAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.ln1(x), mask))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

block = CustomTransformerBlock(256, 4)
z = block(x)
print("Block output shape:", z.shape)

## Part 3: Causal Mask for Autoregressive LM

---

In [ ]:
def causal_mask(seq_len, device):
    return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).unsqueeze(0).unsqueeze(0).to(device)

mask = causal_mask(10, x.device)
out_causal = block(x, mask=mask)
print("With causal mask: no peeking at future tokens.")

## Exercises

1. Add a second block and an embedding + LM head to form a tiny causal LM.
2. Try different d_ff (e.g. 4*d_model) and compare parameter count and speed.
3. Implement a simple "linear attention" variant and compare memory vs standard attention.

---

## Key Takeaways

1. Custom attention = Q,K,V projections, scale, softmax, then V; multi-head splits along feature dim.
2. Pre-LN (norm before attention/FFN) is standard; residual + norm stabilizes training.
3. Causal mask ensures autoregressive property for language modeling.

---

## Next Lesson

**L33: Distributed Training** – Data and model parallelism, multi-GPU.

---